# Trabajo Práctico N° 1: Redes Neuronales CBOW para PLN
**Asignatura**: Aprendizaje Automático Avanzado  
**Carrera**: Tecnicatura Universitaria en Inteligencia Artificial  
**Institución**: Universidad Nacional de Hurlingham (UNAHUR)  
**Profesor**: Juan Miguel Santos (LIDEC)  

---

## Descripción del Proyecto
Este Jupyter Notebook implementa el modelo de representación contextual de palabras **CBOW (Continuous Bag-of-Words)** utilizando **Python y CuPy GPU / PyTorch GPU** para aceleración en placas de video NVIDIA.

La implementación se adhiere estrictamente al material teórico y a la formulación matemática expuesta por la cátedra:
- **Entrada**: Contexto de $C = 2 \times W$ palabras ($W$ a la izquierda, $W$ a la derecha).
- **Capa Oculta**: Activación lineal $h = \frac{1}{C} \sum_{c=1}^C W_{I_c, :}^T$
- **Activación y Salida**: Activaciones lineales $u = h W'$, Softmax $y_j = \frac{\exp(u_j - \max(u))}{\sum \exp(u_{j'} - \max(u))}$
- **Actualización de Gradientes**: Error de salida $e = y - t$, Error oculto $E_H = e W'^T$, $W' \leftarrow W' - \frac{\eta}{B} h^T e$, $W_{I_c, :} \leftarrow W_{I_c, :} - \frac{\eta}{B C} E_{H, b}^T$
- **Token Especial**: Únicamente `<UNK>`.
- **Evaluación de Similaridad**: Producto Interno por defecto, Coseno opcional.
- **Selección de Vocabulario**: Elección excluyente entre `cantidad_palabras_unicas` O `porcentaje_palabras_unicas` con muestreo aleatorio reproducible (`semilla_aleatoria = 26`).
- **Reanudación por Épocas Adicionales**: Método `entrenar_adicional(epocas_adicionales=N)`.

## Sección 1: Configuración de Parámetros y Entorno

Los hiperparámetros y la configuración del proyecto se gestionan de forma centralizada en el archivo [`configuracion.yaml`](configuracion.yaml).

### Parámetros Principales Configurados en `configuracion.yaml`:
- **`ruta_corpus`**: Ruta al archivo del corpus de texto (ej. `datos/corpus_ApAvAu.txt`).
- **`motor_computo`**: Motor de ejecución acelerada en GPU NVIDIA (`"cupy"` o `"pytorch"`).
- **`criterio_seleccion_vocabulario`**: Criterio excluyente (`"cantidad"` o `"porcentaje"`).
- **`cantidad_palabras_unicas` / `porcentaje_palabras_unicas`**: Tamaño del vocabulario seleccionado al azar con semilla `26`.
- **`tamanio_ventana`**: Ventana $W$ de palabras a izquierda y derecha ($W = 4$).
- **`dimension_embedding`**: Dimensión de la capa oculta ($N = 100$).
- **`tasa_aprendizaje`**: Tasa de aprendizaje $\eta = 0.025$.
- **`cantidad_epocas`**: Número de épocas a entrenar ($10$).
- **`frecuencia_respaldo`**: Autoguardado de checkpoints cada $K$ épocas ($2$).
- **`tamanio_lote`**: Tamaño de mini-lote $B$ para procesamiento vectorizado en GPU ($2048$).
- **`tipo_similaridad`**: Criterio de similaridad para palabras (`"producto_interno"` o `"coseno"`).

In [ ]:
import sys
from pathlib import Path

# Asegurar acceso a los módulos del directorio codigo/
ruta_proyecto = Path.cwd()
if str(ruta_proyecto) not in sys.path:
    sys.path.append(str(ruta_proyecto))

from codigo.configuracion import Configuracion
from codigo.tokenizador import Tokenizador
from codigo.generador_vocabulario import GeneradorVocabulario
from codigo.modelo_cbow_cupy import ModeloCbowCuPy, USAR_CUPY
from codigo.modelo_cbow_pytorch import ModeloCbowPyTorch
from codigo.entrenador_cbow import EntrenadorCbow
from codigo.evaluador_similaridad import EvaluadorSimilaridad

# Cargar configuracion centralizada desde configuracion.yaml
config = Configuracion("configuracion.yaml")

print("=== Parámetros Cargados desde configuracion.yaml ===")
diccionario_params = config.a_diccionario()
for clave, valor in diccionario_params.items():
    print(f" - {clave}: {valor}")

print(f"\nAceleración GPU (CuPy disponible): {USAR_CUPY}")

## Sección 2: Carga y Preprocesamiento del Corpus
Cargamos el archivo de texto especificado en `ruta_corpus` y realizamos la tokenización por palabras utilizando expresiones regulares (respetando minúsculas, tildes y caracteres en español).

In [ ]:
# Instanciar tokenizador
tokenizador = Tokenizador(estrategia=config.estrategia_tokenizacion)
ruta_corpus = Path(config.ruta_corpus)

print(f"Leyendo y tokenizando corpus desde: {ruta_corpus}...")
tokens_corpus = tokenizador.tokenizar_archivo(ruta_corpus)
print(f"Total de palabras/tokens en el corpus: {len(tokens_corpus):,}")
print(f"Muestra de primeros 20 tokens: {tokens_corpus[:20]}")

## Sección 3: Construcción del Vocabulario (Muestreo Aleatorio Excluyente)
Se construye el vocabulario seleccionando palabras al azar (semilla 26) y eligiendo EXCLUSIVAMENTE entre `cantidad_palabras_unicas` O `porcentaje_palabras_unicas` según `criterio_seleccion_vocabulario` ("cantidad" o "porcentaje").

In [ ]:
# Instanciar generador de vocabulario
vocabulario = GeneradorVocabulario(token_desconocido=config.token_desconocido)

# Construir vocabulario especificando el criterio deseado y la semilla aleatoria
vocabulario.construir_vocabulario(
    lista_tokens=tokens_corpus,
    criterio_seleccion=config.criterio_seleccion_vocabulario,
    cantidad_palabras_unicas=config.cantidad_palabras_unicas,
    porcentaje_palabras_unicas=config.porcentaje_palabras_unicas,
    frecuencia_minima=config.frecuencia_minima,
    semilla_aleatoria=config.semilla_aleatoria,
)

# Convertir la secuencia de palabras del corpus a una lista de indices enteros
indices_corpus = vocabulario.convertir_tokens_a_indices(tokens_corpus)
print(f"Total de indices procesados: {len(indices_corpus):,}")
print(f"Primeros 15 indices: {indices_corpus[:15]}")

## Sección 4: Entrenamiento del Modelo CBOW
Única celda de entrenamiento. Utiliza el tamaño de ventana de contexto $W$ especificado en `configuracion.yaml` (`config.tamanio_ventana`).  
El nombre del modelo toma por defecto el valor de la ventana configurada (ej. `modelo_w4`).

In [ ]:
# Obtener el tamaño de ventana configurado en configuracion.yaml
tamanio_w = config.tamanio_ventana
tamanio_v = vocabulario.tamanio_vocabulario

# Nombre del modelo (por defecto basado en la ventana W configurada)
nombre_modelo = f"modelo_w{tamanio_w}"

print(f"=== Iniciando Entrenamiento del Modelo: '{nombre_modelo}' (Ventana W = {tamanio_w}) ===")

# Instanciar el modelo CBOW segun el motor configurado ("cupy" o "pytorch")
if config.motor_computo == "pytorch":
    modelo_cbow = ModeloCbowPyTorch(
        tamanio_vocabulario=tamanio_v,
        dimension_embedding=config.dimension_embedding,
        semilla_aleatoria=config.semilla_aleatoria,
    )
else:
    modelo_cbow = ModeloCbowCuPy(
        tamanio_vocabulario=tamanio_v,
        dimension_embedding=config.dimension_embedding,
        semilla_aleatoria=config.semilla_aleatoria,
    )

# Instanciar el entrenador
entrenador_cbow = EntrenadorCbow(
    modelo=modelo_cbow,
    tasa_aprendizaje=config.tasa_aprendizaje,
    directorio_respaldos=config.directorio_respaldos,
    frecuencia_respaldo=config.frecuencia_respaldo,
)

# Entrenar modelo
historial_perdida = entrenador_cbow.entrenar(
    indices_tokens=indices_corpus,
    tamanio_ventana=tamanio_w,
    tamanio_lote=config.tamanio_lote,
    cantidad_epocas=config.cantidad_epocas,
)

### Sección 4.1: Reanudar / Continuar el Entrenamiento por Épocas Adicionales
Para reanudar un entrenamiento existente desde un archivo de resguardo `.npz` y continuar por $N$ épocas adicionales, se carga el archivo `.npz` y se invoca el método `entrenar_adicional(epocas_adicionales=N)`.

In [ ]:
# --- EJEMPLO DE CONTINUACIÓN POR ÉPOCAS ADICIONALES ---
# 1. Instanciar un modelo nuevo con la misma dimensión de embedding
modelo_reanudado = ModeloCbowCuPy(
    tamanio_vocabulario=vocabulario.tamanio_vocabulario,
    dimension_embedding=config.dimension_embedding,
)

# 2. Cargar los pesos guardados desde el archivo de backup (ejemplo: epoca 10)
ruta_backup = Path(config.directorio_respaldos) / f"modelo_cbow_w{config.tamanio_ventana}_epoca_10.npz"
if ruta_backup.exists():
    epoca_guardada, historial_anterior = modelo_reanudado.cargar_modelo(ruta_backup)
    
    # 3. Instanciar el entrenador y restaurar el historial de pérdidas anteriores
    entrenador_reanudado = EntrenadorCbow(
        modelo=modelo_reanudado,
        tasa_aprendizaje=config.tasa_aprendizaje,
        directorio_respaldos=config.directorio_respaldos,
        frecuencia_respaldo=config.frecuencia_respaldo,
    )
    entrenador_reanudado.historial_perdida = list(historial_anterior)
    entrenador_reanudado.epoca_actual_cargada = epoca_guardada
    
    # 4. Entrenar N épocas adicionales (ejemplo: 5 épocas adicionales)
    perdida_actualizada = entrenador_reanudado.entrenar_adicional(
        indices_tokens=indices_corpus,
        epocas_adicionales=5,
        tamanio_ventana=config.tamanio_ventana,
        tamanio_lote=config.tamanio_lote,
    )
else:
    print(f"El archivo {ruta_backup} no se encuentra todavía en la carpeta respaldos/.")

## Sección 5: Evaluación de Similaridad de Palabras (Producto Interno vs Coseno)
Evaluamos la similaridad de una palabra objetivo ingresada por el usuario contra todas las palabras del vocabulario.  
Por defecto se utiliza **Producto Interno**, con la opción de activar **Similaridad de Coseno**.

In [ ]:
# Instanciar evaluador con el modelo entrenado
evaluador = EvaluadorSimilaridad(modelo_cbow, vocabulario)

# Palabra ingresada para la comparación
palabra_evaluar = "hombre"
top_k = 10

print(f"=== Busqueda por PRODUCTO INTERNO para la palabra '{palabra_evaluar}' ===\n")
similares_pi = evaluador.buscar_palabras_similares(
    palabra_evaluar,
    top_k=top_k,
    tipo_similaridad="producto_interno",
)

posicion = 1
for par in similares_pi:
    p, s = par
    print(f"  {posicion:2d}. {p:<15} Puntaje: {s:.4f}")
    posicion += 1

print(f"\n=== Busqueda por SIMILARIDAD DE COSENO para la palabra '{palabra_evaluar}' ===\n")
similares_cos = evaluador.buscar_palabras_similares(
    palabra_evaluar,
    top_k=top_k,
    tipo_similaridad="coseno",
)

posicion = 1
for par in similares_cos:
    p, s = par
    print(f"  {posicion:2d}. {p:<15} Coseno: {s:.4f}")
    posicion += 1

## Sección 6: Comparación entre Dos Modelos o Resguardos (.npz)
Esta celda permite cargar y comparar las curvas de pérdida y similaridad de palabras entre dos modelos o resguardos `.npz` indicando sus rutas de archivo.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# --- RUTAS DE LOS DOS MODELOS A COMPARAR ---
ruta_modelo_1 = Path(config.directorio_respaldos) / "modelo_cbow_w4_epoca_10.npz"
ruta_modelo_2 = Path(config.directorio_respaldos) / "modelo_cbow_w5_epoca_10.npz"

print('=== Comparación de Modelos ===')
print(f'Modelo 1: {ruta_modelo_1}')
print(f'Modelo 2: {ruta_modelo_2}\n')

loss_m1 = []
loss_m2 = []

if ruta_modelo_1.exists():
    mod_1 = ModeloCbowCuPy(
        tamanio_vocabulario=vocabulario.tamanio_vocabulario,
        dimension_embedding=config.dimension_embedding,
    )
    ep_1, loss_m1 = mod_1.cargar_modelo(ruta_modelo_1)
else:
    print(f"El archivo {ruta_modelo_1} no existe todavía.")

if ruta_modelo_2.exists():
    mod_2 = ModeloCbowCuPy(
        tamanio_vocabulario=vocabulario.tamanio_vocabulario,
        dimension_embedding=config.dimension_embedding,
    )
    ep_2, loss_m2 = mod_2.cargar_modelo(ruta_modelo_2)
else:
    print(f"El archivo {ruta_modelo_2} no existe todavía.")

# Graficar curvas de pérdida comparativas si hay datos
if len(loss_m1) > 0 or len(loss_m2) > 0:
    sns.set_theme(style="whitegrid")
    plt.figure(figsize=(10, 5))
    
    if len(loss_m1) > 0:
        epocas_m1 = []
        for idx in range(1, len(loss_m1) + 1):
            epocas_m1.append(idx)
        plt.plot(epocas_m1, loss_m1, marker="o", color="#1f77b4", linewidth=2, label=f"Modelo 1 ({ruta_modelo_1.name})")
        
    if len(loss_m2) > 0:
        epocas_m2 = []
        for idx in range(1, len(loss_m2) + 1):
            epocas_m2.append(idx)
        plt.plot(epocas_m2, loss_m2, marker="s", color="#ff7f0e", linewidth=2, label=f"Modelo 2 ({ruta_modelo_2.name})")

    plt.title("Evolución Comparativa de Pérdida Promedio por Época", fontsize=14, fontweight="bold")
    plt.xlabel("Época", fontsize=12)
    plt.ylabel("Pérdida Promedio", fontsize=12)
    plt.legend(fontsize=11)
    plt.tight_layout()
    plt.show()

## Sección 7: Conclusiones y Registro en Bitácora

*(Esta sección se encuentra libre para completar con tus conclusiones finales y hallazgos principales tras la ejecución de los experimentos)*

- **Conclusión 1**:
- **Conclusión 2**:
- **Conclusión 3**: